In [0]:
!find . -type d -name "__pycache__" -exec rm -r {} +

In [0]:
%pip install pyspark pandas psycopg2-binary

In [0]:
%restart_python

In [0]:
import sys
from pathlib import Path

# Ajusta la ruta a tu estructura de carpetas:
PROJECT_ROOT = Path.cwd().parent  
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Directorio raíz del proyecto agregado:", PROJECT_ROOT)
from m01_data_ingestion import ingest as ingest_data

# Ejecuta la ingesta
df = ingest_data()
print("Primeras filas del DataFrame crudo:")
display(df.head())

#Prueba Modulos step

### Step01

In [0]:
# Prueba para step01_import: ParquetPartitionLoader

from m05_feature_engineering.step01_import import ParquetPartitionLoader

# Instancia el loader (usa la ruta por defecto configurada en el módulo)
loader = ParquetPartitionLoader()

# Carga los datasets particionados
X_train, X_test, X_backtest, y_train, y_test, y_backtest = loader.load()

# Muestra el shape de cada partición
print(f"X_train:    {X_train.shape}, y_train:    {y_train.shape}")
print(f"X_test:     {X_test.shape},  y_test:     {y_test.shape}")
print(f"X_backtest: {X_backtest.shape}, y_backtest: {y_backtest.shape}")

# Opcional: muestra las primeras filas para validar visualmente
print("\nPrimeras filas de X_train:")
print(X_train.head())
print("\nPrimeras filas de y_train:")
print(y_train.head())


### step02

In [0]:
import pandas as pd
import numpy as np

# 1. Creamos tres dataframes de juguete (pueden ser idénticos)
data = {
    "edad":   [25, np.nan, 30, 29, np.nan],
    "peso":   [70, 68, np.nan, 75, 72],
    "altura": [1.65, 1.7, 1.68, np.nan, 1.72]
}

train_df    = pd.DataFrame(data)
test_df     = pd.DataFrame(data)
backtest_df = pd.DataFrame(data)

print("Datos originales (train):")
print(train_df)

# 2. Prueba de imputación usando tu step02_imputation
from m05_feature_engineering.step02_imputation import SimpleImputerAdapter

imputer = SimpleImputerAdapter()

# Ajustamos/imputamos sobre train
train_imputed = imputer.fit_transform(train_df.copy())
# Imputamos sobre test y backtest
test_imputed = imputer.transform(test_df.copy())
backtest_imputed = imputer.transform(backtest_df.copy())

# 3. Mostramos resultados
print("\nTrain imputado:\n", train_imputed)
print("\nTest imputado:\n", test_imputed)
print("\nBacktest imputado:\n", backtest_imputed)


### step03

In [0]:
import pandas as pd
from m05_feature_engineering.step03_outliers import IQRHandler

# DataFrame base con outliers
data = {
    "edad":   [25, 26, 27, 28, 100],   # 100 es un outlier
    "peso":   [70, 68, 69, 71, 150],   # 150 es un outlier
    "altura": [1.65, 1.70, 1.68, 1.66, 2.5]  # 2.5 es un outlier
}

train_df    = pd.DataFrame(data)
test_df     = pd.DataFrame(data)
backtest_df = pd.DataFrame(data)

print("=== Datos originales ===")
print(train_df)

# Instancia y ajusta sobre train
handler = IQRHandler(factor=1.5)
train_capped = handler.fit_transform(train_df.copy())

# Aplica transform sobre test y backtest
test_capped = handler.transform(test_df.copy())
backtest_capped = handler.transform(backtest_df.copy())

print("\n=== Train tras capping ===")
print(train_capped)

print("\n=== Test tras capping ===")
print(test_capped)

print("\n=== Backtest tras capping ===")
print(backtest_capped)


### step04

In [0]:
import pandas as pd
from m05_feature_engineering.step04_transformation import StandardScaleTransformer

# 1. Creamos dataframes de juguete
data = {
    "edad":   [20, 30, 40, 60, 100],
    "ingresos": [1000, 1200, 2000, 5000, 15000]
}
train_df    = pd.DataFrame(data)
test_df     = pd.DataFrame(data)
backtest_df = pd.DataFrame(data)

print("=== Datos originales (train) ===")
print(train_df)

# 2. Instanciamos y aplicamos el transformador
transformer = StandardScaleTransformer()

# Ajustamos sobre train
train_trans = transformer.fit_transform(train_df.copy())

# Transformamos test y backtest usando los parámetros aprendidos en train
test_trans     = transformer.transform(test_df.copy())
backtest_trans = transformer.transform(backtest_df.copy())

print("\n=== Train transformado ===")
print(train_trans)

print("\n=== Test transformado ===")
print(test_trans)

print("\n=== Backtest transformado ===")
print(backtest_trans)


### step05 -> step08

In [0]:
import pandas as pd
from m05_feature_engineering.step05_encoding  import OneHotEncoderAdapter
from m05_feature_engineering.step06_feature_gen import PolynomialFeatureGenerator
from m05_feature_engineering.step07_metrics    import JsonMetricsExporter
from m05_feature_engineering.step08_drift      import PSIDriftDetector

# ------------------------------------------------------------------
# 1) Creamos DataFrames de juguete (train / test / backtest)
data = {
    "sexo": ["M", "F", "F", "M", "F"],
    "ciudad": ["A", "B", "A", "C", "B"],
    "edad": [25, 30, 35, 28, 40],
    "ingresos": [1000, 1500, 1200, 1100, 3000],
    "target": [0, 1, 0, 0, 1]
}
train_df    = pd.DataFrame(data)
test_df     = pd.DataFrame(data)
backtest_df = pd.DataFrame(data)

# Separamos features y target
X_train, y_train = train_df.drop("target", axis=1), train_df["target"]
X_test,  y_test  = test_df.drop("target", axis=1),  test_df["target"]
X_back,  y_back  = backtest_df.drop("target", axis=1), backtest_df["target"]

# ------------------------------------------------------------------
# 2) One-Hot Encoding
encoder = OneHotEncoderAdapter()
X_train_enc = encoder.fit_transform(X_train.copy())
X_test_enc  = encoder.transform(X_test.copy())
X_back_enc  = encoder.transform(X_back.copy())

print("Columns after OHE:", X_train_enc.columns.tolist())

# ------------------------------------------------------------------
# 3) Generación de polinomios grado 2
poly_gen = PolynomialFeatureGenerator(degree=2)
X_train_poly = poly_gen.fit_transform(X_train_enc.copy())
X_test_poly  = poly_gen.transform(X_test_enc.copy())
X_back_poly  = poly_gen.transform(X_back_enc.copy())

print("\nShape after PolynomialFeatures:")
print("  Train:", X_train_poly.shape, "Test:", X_test_poly.shape)

# ------------------------------------------------------------------
# 4) Exportamos métricas del train transformado
metrics = JsonMetricsExporter()
metrics.export(pd.concat([X_train_poly, y_train], axis=1), "metrics/feature_metrics.json")
print("\n✅ Métricas guardadas en metrics/feature_metrics.json")

# ------------------------------------------------------------------
# 5) Detectamos drift entre train y test
drift_detector = PSIDriftDetector(bins=10)
psi_scores = drift_detector.compute(X_train_poly, X_test_poly)
print("\nPSI por columna:\n", psi_scores)


### pipeline completo

In [0]:
from m05_feature_engineering.pipeline_engineering import run_pipeline
run_pipeline()
from pathlib import Path
import pandas as pd
# 1) Calcula la carpeta data/processed relativa al cwd (src/app)
base = Path.cwd().parent / "data" / "processed"

# 2) Carga cada Parquet
df_train   = pd.read_parquet(base / "X_train_processed.parquet")
df_test    = pd.read_parquet(base / "X_test_processed.parquet")
df_back    = pd.read_parquet(base / "X_backtest_processed.parquet")

# 3) Muestra un vistazo
print("Train procesado:\n", df_train.head(), "\n")
print("Test procesado:\n", df_test.head(), "\n")
print("Backtest procesado:\n", df_back.head())


# Feature selection

In [1]:
from m06__feature_selection.step01_import import ParquetPartitionLoader2

loader = ParquetPartitionLoader2()          # usa la ruta por defecto corregida
X_train, X_test, X_back, y_train, y_test, y_back = loader.load()

print("Shapes:", X_train.shape, X_test.shape, X_back.shape)


Shapes: (235, 15) (67, 15) (34, 15)


In [2]:
# %% [markdown]
# ## Test de step02_filtering.filter_partitions usando loader de step01_import

# %%
import sys
from pathlib import Path

# 1) Asegúrate de que el directorio raíz del proyecto (donde está src/) esté en sys.path
ROOT_DIR = Path().resolve().parent.parent  # ajusta según la ubicación de tu .ipynb
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

# %%
# 2) Importa loader y función a testear
from m06__feature_selection.step01_import import ParquetPartitionLoader2
from m06__feature_selection.step02_filtering import filter_partitions

# %%
# 3) Carga los datos con el loader (mismo comportamiento que tu test de step01)
loader = ParquetPartitionLoader2()            # usa ruta y particiones por defecto
X_train, X_test, X_back, y_train, y_test, y_back = loader.load()

print("🚀 Shapes originales:", 
      "X_train", X_train.shape, 
      "X_test",  X_test.shape, 
      "X_back",  X_back.shape)

# Opcional: inspección rápida de columnas
print("Columnas originales:", X_train.columns.tolist())

# %%
# 4) Aplica el filtrado de features
X_tr_f, X_te_f, X_ba_f, pipeline = filter_partitions(
    X_train, X_test, X_back, y_train
)

# %%
# 5) Resultados
print("✅ Shapes filtrados:", 
      "X_train_f", X_tr_f.shape, 
      "X_test_f",  X_te_f.shape, 
      "X_back_f",  X_ba_f.shape)

print("Columnas filtradas:", X_tr_f.columns.tolist())

# %%
# 6) (Opcional) Verifica que el pipeline funcione en test y back
X_te_check = pipeline.transform(X_test)
X_ba_check = pipeline.transform(X_back)

assert list(X_te_check.columns) == list(X_tr_f.columns), "Columnas desalineadas en test"
assert list(X_ba_check.columns) == list(X_tr_f.columns), "Columnas desalineadas en back"

print("👍 El pipeline transforma correctamente test y back con las mismas columnas.")

# %%
# 7) Muestra un par de filas filtradas
X_tr_f.head()


🚀 Shapes originales: X_train (235, 15) X_test (67, 15) X_back (34, 15)
Columnas originales: ['Semana', 'EPS_A', 'EPS_B', 'EPS_C', 'EPS_D', 'EPS_A^2', 'EPS_A EPS_B', 'EPS_A EPS_C', 'EPS_A EPS_D', 'EPS_B^2', 'EPS_B EPS_C', 'EPS_B EPS_D', 'EPS_C^2', 'EPS_C EPS_D', 'EPS_D^2']
✅ Shapes filtrados: X_train_f (235, 15) X_test_f (67, 15) X_back_f (34, 15)
Columnas filtradas: ['EPS_A', 'EPS_B', 'EPS_C', 'EPS_D', 'EPS_A^2', 'EPS_A EPS_B', 'EPS_A EPS_C', 'EPS_A EPS_D', 'EPS_B^2', 'EPS_B EPS_C', 'EPS_B EPS_D', 'EPS_C^2', 'EPS_C EPS_D', 'EPS_D^2', 'Semana']
👍 El pipeline transforma correctamente test y back con las mismas columnas.


,EPS_A,EPS_B,EPS_C,EPS_D,EPS_A^2,EPS_A EPS_B,EPS_A EPS_C,EPS_A EPS_D,EPS_B^2,EPS_B EPS_C,EPS_B EPS_D,EPS_C^2,EPS_C EPS_D,EPS_D^2,Semana
0,-0.659360,-0.810316,0.052051,-0.008460,0.434756,0.534290,-0.034320,0.005579,0.656611,-0.042177,0.006856,0.002709,-0.000440,0.000072,2019-02-10
1,0.433301,0.841604,0.274329,-2.119669,0.187750,0.364668,0.118867,-0.918455,0.708297,0.230876,-1.783922,0.075256,-0.581487,4.492997,2019-02-17
2,-0.659360,-1.185150,-0.406177,-0.769008,0.434756,0.781441,0.267817,0.507054,1.404580,0.481380,0.911390,0.164979,0.312353,0.591374,2019-02-24
3,0.253322,0.478917,-1.649742,-0.258673,0.064172,0.121320,-0.417916,-0.065528,0.229362,-0.790090,-0.123883,2.721647,0.426743,0.066912,2019-03-03
4,0.968728,0.296694,-0.642804,0.964606,0.938433,0.287416,-0.622702,0.934440,0.088027,-0.190716,0.286193,0.413197,-0.620052,0.930464,2019-03-10


In [2]:
# %%
#import warnings
#warnings.simplefilter("ignore", FutureWarning)

import sys
from pathlib import Path

# 1) Asegura que el proyecto (donde está src/) esté en sys.path
PROJECT_ROOT = Path().resolve().parents[1]  # si el notebook está en src/app/tests
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# %%
# 2) Importa los tres módulos
from m06__feature_selection.step01_import   import ParquetPartitionLoader2
from m06__feature_selection.step02_filtering import filter_partitions
from m06__feature_selection.step03_frame     import frame_partitions

# %%
# 3) Carga particiones con step01
loader = ParquetPartitionLoader2()
X_train, X_test, X_back, y_train, y_test, y_back = loader.load()
print("🚀 Shapes originales:",
      "X_train", X_train.shape,
      "X_test",  X_test.shape,
      "X_back",  X_back.shape)

# %%
# 4) Aplica filtro rough con step02
X_tr_f, X_te_f, X_ba_f, rough_tf = filter_partitions(
    X_train, X_test, X_back, y_train
)
print("✅ After rough filter:",
      "X_tr_f", X_tr_f.shape,
      "X_te_f", X_te_f.shape,
      "X_ba_f", X_ba_f.shape)
print("→ columnas after rough:", rough_tf.filter.selected_cols + rough_tf.non_num_cols)

# %% [markdown]
# ## 5. Prueba de `frame_partitions` para clasificación
# Por defecto usa `LogisticRegression` y selecciona las `forward_k=“auto”` mejores.

# %%
#Xc_tr_f, Xc_te_f, Xc_ba_f, selector_clf = frame_partitions(
#    X_tr_f, X_te_f, X_ba_f, y_train
#)
#print("🔍 Clasificación → columnas seleccionadas:", selector_clf.selected_cols)
#print("Shapes clasificación:",
#      Xc_tr_f.shape, Xc_te_f.shape, Xc_ba_f.shape)

# %% [markdown]
# ## 6. Prueba de `frame_partitions` para regresión
# Usamos `LinearRegression`, elegimos 4 en forward y dejamos 2 al final.

# %%
from sklearn.linear_model import LinearRegression

Xr_tr_f, Xr_te_f, Xr_ba_f, selector_reg = frame_partitions(
    X_tr_f, X_te_f, X_ba_f, y_train,
    estimator=LinearRegression(),
    forward_k=4,
    final_k=2,
)
print("🔍 Regresión → columnas seleccionadas:", selector_reg.selected_cols)
print("Shapes regresión:",
      Xr_tr_f.shape, Xr_te_f.shape, Xr_ba_f.shape)

# %%
# 7) Inspección rápida: primeras filas del set regresión
display(Xr_tr_f.head())


🚀 Shapes originales: X_train (235, 15) X_test (67, 15) X_back (34, 15)
✅ After rough filter: X_tr_f (235, 15) X_te_f (67, 15) X_ba_f (34, 15)
→ columnas after rough: ['EPS_A', 'EPS_B', 'EPS_C', 'EPS_D', 'EPS_A^2', 'EPS_A EPS_B', 'EPS_A EPS_C', 'EPS_A EPS_D', 'EPS_B^2', 'EPS_B EPS_C', 'EPS_B EPS_D', 'EPS_C^2', 'EPS_C EPS_D', 'EPS_D^2', 'Semana']
🔍 Regresión → columnas seleccionadas: ['EPS_A', 'EPS_B']
Shapes regresión: (235, 3) (67, 3) (34, 3)


,EPS_A,EPS_B,Semana
0,-0.659360,-0.810316,2019-02-10
1,0.433301,0.841604,2019-02-17
2,-0.659360,-1.185150,2019-02-24
3,0.253322,0.478917,2019-03-03
4,0.968728,0.296694,2019-03-10


In [3]:
%pip install abess

Note: you may need to restart the kernel to use updated packages.


In [4]:
%restart_python

UsageError: Line magic function `%restart_python` not found.


In [2]:
# %% [markdown]
# # Test integrado de los pasos 2, 3 y 4 sobre particiones reales

# %%
import sys
from pathlib import Path

# 1) Asegura que la raíz del proyecto esté en sys.path
PROJECT_ROOT = Path().resolve().parents[1]  # ajusta según la ubicación de tu notebook
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# %%
# 2) Importa todos los módulos a testear
from m06__feature_selection.step01_import   import ParquetPartitionLoader2
from m06__feature_selection.step02_filtering import filter_partitions
from m06__feature_selection.step03_frame     import frame_partitions
from m06__feature_selection.step04_abess     import abess_partitions

# %% [markdown]
# ## 3) Carga las particiones reales con el loader de step01_import

# %%
loader = ParquetPartitionLoader2()
X_train, X_test, X_back, y_train, y_test, y_back = loader.load()
print("🚀 Shapes originales:",
      "X_train", X_train.shape,
      "X_test",  X_test.shape,
      "X_back",  X_back.shape)

# %% [markdown]
# ## 4) Paso 2: filtro rough (step02_filtering)

# %%
X_tr_f, X_te_f, X_ba_f, rough_tf = filter_partitions(
    X_train, X_test, X_back, y_train
)
print("✅ After rough filter:",
      "X_tr_f", X_tr_f.shape,
      "X_te_f", X_te_f.shape,
      "X_ba_f", X_ba_f.shape)
print("→ columnas after rough:", rough_tf.filter.selected_cols + rough_tf.non_num_cols)

# %% [markdown]
# ## 5) Paso 3: selección secuencial + RFE (step03_frame)

# %%
from sklearn.linear_model import LinearRegression

X_tr_fr, X_te_fr, X_ba_fr, frame_sel = frame_partitions(
    X_tr_f, X_te_f, X_ba_f, y_train,
    estimator=LinearRegression(), forward_k=4, final_k=2
)
print("🔍 Frame selection → columnas seleccionadas:", frame_sel.selected_cols)
print("Shapes frame sel:",
      X_tr_fr.shape, X_te_fr.shape, X_ba_fr.shape)

# %% [markdown]
# ## 6) Paso 4: selección con ABESS (step04_abess) en modo regresión

# %%
X_tr_ab, X_te_ab, X_ba_ab, abess_sel = abess_partitions(
    X_tr_fr, X_te_fr, X_ba_fr, y_train,
    mode="regression"
)
print("✨ ABESS selection → columnas seleccionadas:", abess_sel.selected_cols)
print("Shapes abess sel:",
      X_tr_ab.shape, X_te_ab.shape, X_ba_ab.shape)

# %% [markdown]
# ## 7) Inspección final de las particiones seleccionadas

# %%
print("Primeras filas de X_train tras ABESS:")
display(X_tr_ab.head())


🚀 Shapes originales: X_train (235, 15) X_test (67, 15) X_back (34, 15)
✅ After rough filter: X_tr_f (235, 15) X_te_f (67, 15) X_ba_f (34, 15)
→ columnas after rough: ['EPS_A', 'EPS_B', 'EPS_C', 'EPS_D', 'EPS_A^2', 'EPS_A EPS_B', 'EPS_A EPS_C', 'EPS_A EPS_D', 'EPS_B^2', 'EPS_B EPS_C', 'EPS_B EPS_D', 'EPS_C^2', 'EPS_C EPS_D', 'EPS_D^2', 'Semana']
🔍 Frame selection → columnas seleccionadas: ['EPS_A', 'EPS_B']
Shapes frame sel: (235, 3) (67, 3) (34, 3)
✨ ABESS selection → columnas seleccionadas: ['EPS_A']
Shapes abess sel: (235, 2) (67, 2) (34, 2)
Primeras filas de X_train tras ABESS:


,EPS_A,Semana
0,-0.659360,2019-02-10
1,0.433301,2019-02-17
2,-0.659360,2019-02-24
3,0.253322,2019-03-03
4,0.968728,2019-03-10


# SOLO PASO shap

In [1]:
# %% Prueba robusta de shap_partitions (step05_shap_select)

import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Ajusta src/app en sys.path
APP_DIR = Path().resolve()
if str(APP_DIR) not in sys.path:
    sys.path.append(str(APP_DIR))

# Importa función
from m06__feature_selection.step05_shap_select import shap_partitions

# Simulación de datos de regresión (pacientes nuevos semanales)
rng = np.random.default_rng(42)
weeks = 100
X = pd.DataFrame({
    "clima": rng.normal(size=weeks),
    "publicidad": rng.integers(0, 2, weeks),
    "eventos": rng.poisson(0.5, weeks),
    "promociones": rng.integers(0, 2, weeks),
    "competidores": rng.poisson(2, weeks),
})
y = (
    25 + 3*X["clima"] + 8*X["publicidad"] 
    - 2*X["competidores"] + rng.normal(0, 2, weeks)
)

# División (60% train, 20% test, 20% backtest)
idx1, idx2 = int(weeks*0.6), int(weeks*0.8)
X_train, y_train = X[:idx1], y[:idx1]
X_test, X_back = X[idx1:idx2], X[idx2:]

# Ejecuta selección SHAP en modo regresión
X_tr_sel, X_te_sel, X_ba_sel, selector = shap_partitions(
    X_train, X_test, X_back, y_train, 
    top_n=3, task="regression"
)

# Resultados
print("Features seleccionados por SHAP:", selector.selected_cols)
print("Shapes:", X_tr_sel.shape, X_te_sel.shape, X_ba_sel.shape)
display(X_tr_sel.head())


/home/roma/proyecto_mlops/ambiente_python/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Features seleccionados por SHAP: ['publicidad', 'competidores', 'clima']
Shapes: (60, 3) (20, 3) (20, 3)


,publicidad,competidores,clima
0,0,3,0.304717
1,0,5,-1.039984
2,1,0,0.750451
3,1,0,0.940565
4,0,4,-1.951035


In [2]:

import sys
from pathlib import Path

# 1) Asegura que la raíz del proyecto esté en sys.path
PROJECT_ROOT = Path().resolve().parents[1]  # ajusta si tu notebook está en src/app/tests
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# %%
# 2) Importa loader y todos los pasos
from m06__feature_selection.step01_import   import ParquetPartitionLoader2
from m06__feature_selection.step02_filtering import filter_partitions
from m06__feature_selection.step03_frame     import frame_partitions
from m06__feature_selection.step04_abess     import abess_partitions
from m06__feature_selection.step05_shap_select import shap_partitions

# %% [markdown]
# ## 3) Carga las particiones reales con el loader de step01_import

# %%
loader = ParquetPartitionLoader2()
X_train, X_test, X_back, y_train, y_test, y_back = loader.load()
print("🚀 Shapes originales:",
      "X_train", X_train.shape,
      "X_test",  X_test.shape,
      "X_back",  X_back.shape)

# %% [markdown]
# ## 4) step02_filtering: filtro rough

# %%
X_tr_f, X_te_f, X_ba_f, rough_tf = filter_partitions(
    X_train, X_test, X_back, y_train
)
print("✅ After rough filter:",
      "X_tr_f", X_tr_f.shape,
      "X_te_f", X_te_f.shape,
      "X_ba_f", X_ba_f.shape)
print("→ columnas after rough:",
      rough_tf.filter.selected_cols + rough_tf.non_num_cols)

# %% [markdown]
# ## 5) step03_frame: selección forward + RFE (regresión)

# %%
from sklearn.linear_model import LinearRegression

X_tr_fr, X_te_fr, X_ba_fr, frame_sel = frame_partitions(
    X_tr_f, X_te_f, X_ba_f, y_train,
    estimator=LinearRegression(), forward_k=4, final_k=2
)
print("🔍 Frame selection → columnas seleccionadas:",
      frame_sel.selected_cols)
print("Shapes frame sel:",
      X_tr_fr.shape, X_te_fr.shape, X_ba_fr.shape)

# %% [markdown]
# ## 6) step04_abess: selección por ABESS (modo regresión)

# %%
X_tr_ab, X_te_ab, X_ba_ab, abess_sel = abess_partitions(
    X_tr_fr, X_te_fr, X_ba_fr, y_train,
    mode="regression"
)
print("✨ ABESS selection → columnas seleccionadas:",
      abess_sel.selected_cols)
print("Shapes abess sel:",
      X_tr_ab.shape, X_te_ab.shape, X_ba_ab.shape)

# %% [markdown]
# ## 7) step05_shap_select: selección por valores SHAP (modo regresión)

# %%
X_tr_sh, X_te_sh, X_ba_sh, shap_sel = shap_partitions(
    X_tr_ab, X_te_ab, X_ba_ab, y_train,
    top_n=3, task="regression"
)
print("🌟 SHAP selection → columnas seleccionadas:",
      shap_sel.selected_cols)
print("Shapes SHAP sel:",
      X_tr_sh.shape, X_te_sh.shape, X_ba_sh.shape)

# %% [markdown]
# ## 8) Inspección final de las particiones SHAP

# %%
print("Primeras filas de X_train tras SHAP:")
display(X_tr_sh.head())


🚀 Shapes originales: X_train (235, 15) X_test (67, 15) X_back (34, 15)
✅ After rough filter: X_tr_f (235, 15) X_te_f (67, 15) X_ba_f (34, 15)
→ columnas after rough: ['EPS_A', 'EPS_B', 'EPS_C', 'EPS_D', 'EPS_A^2', 'EPS_A EPS_B', 'EPS_A EPS_C', 'EPS_A EPS_D', 'EPS_B^2', 'EPS_B EPS_C', 'EPS_B EPS_D', 'EPS_C^2', 'EPS_C EPS_D', 'EPS_D^2', 'Semana']
🔍 Frame selection → columnas seleccionadas: ['EPS_A', 'EPS_B']
Shapes frame sel: (235, 3) (67, 3) (34, 3)
✨ ABESS selection → columnas seleccionadas: ['EPS_A']
Shapes abess sel: (235, 2) (67, 2) (34, 2)
🌟 SHAP selection → columnas seleccionadas: ['EPS_A']
Shapes SHAP sel: (235, 2) (67, 2) (34, 2)
Primeras filas de X_train tras SHAP:


,EPS_A,Semana
0,-0.659360,2019-02-10
1,0.433301,2019-02-17
2,-0.659360,2019-02-24
3,0.253322,2019-03-03
4,0.968728,2019-03-10


In [2]:
# # Test completo de feature selection: steps 2 → 3 → 4 → 5 → 6

# %%
import sys
from pathlib import Path

# Asegura que la raíz del proyecto esté en sys.path
PROJECT_ROOT = Path().resolve().parents[1]  # ajusta si tu notebook está en src/app/tests
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# %%
# 1) Importa loader y todos los pasos
from m06__feature_selection.step01_import    import ParquetPartitionLoader2
from m06__feature_selection.step02_filtering import filter_partitions
from m06__feature_selection.step03_frame     import frame_partitions
from m06__feature_selection.step04_abess     import abess_partitions
from m06__feature_selection.step05_shap_select import shap_partitions
from m06__feature_selection.step06_permutation import permutation_partitions

# %%
# 2) Carga particiones reales
loader = ParquetPartitionLoader2()
X_train, X_test, X_back, y_train, y_test, y_back = loader.load()
print("🚀 Shapes originales:",
      "X_train", X_train.shape,
      "X_test",  X_test.shape,
      "X_back",  X_back.shape)

# %%
# 3) step02_filtering: elimina constantes y correlaciones altas
X_tr_f, X_te_f, X_ba_f, rough_tf = filter_partitions(
    X_train, X_test, X_back, y_train
)
print("✅ After rough filter:",
      X_tr_f.shape, X_te_f.shape, X_ba_f.shape)
print("→ cols after rough:", rough_tf.filter.selected_cols + rough_tf.non_num_cols)

# %%
# 4) step03_frame: selección forward + RFE (regresión)
from sklearn.linear_model import LinearRegression

X_tr_fr, X_te_fr, X_ba_fr, frame_sel = frame_partitions(
    X_tr_f, X_te_f, X_ba_f, y_train,
    estimator=LinearRegression(), forward_k=4, final_k=2
)
print("🔍 Frame sel cols:", frame_sel.selected_cols)
print("Shapes frame sel:", X_tr_fr.shape, X_te_fr.shape, X_ba_fr.shape)

# %%
# 5) step04_abess: selección ABESS / LassoCV (regresión)
X_tr_ab, X_te_ab, X_ba_ab, abess_sel = abess_partitions(
    X_tr_fr, X_te_fr, X_ba_fr, y_train,
    mode="regression"
)
print("✨ ABESS sel cols:", abess_sel.selected_cols)
print("Shapes abess sel:", X_tr_ab.shape, X_te_ab.shape, X_ba_ab.shape)

# %%
# 6) step05_shap_select: top_n por importancia SHAP (regresión)
X_tr_sh, X_te_sh, X_ba_sh, shap_sel = shap_partitions(
    X_tr_ab, X_te_ab, X_ba_ab, y_train,
    top_n=3, task="regression"
)
print("🌟 SHAP sel cols:", shap_sel.selected_cols)
print("Shapes SHAP sel:", X_tr_sh.shape, X_te_sh.shape, X_ba_sh.shape)

# %%
# 7) step06_permutation: importancia por permutación (regresión)
X_tr_perm, X_te_perm, X_ba_perm, perm_sel = permutation_partitions(
    X_tr_sh, X_te_sh, X_ba_sh, y_train,
    tol=0.01, task="regression", scoring="r2", n_repeats=10
)
print("🔁 Permutation sel cols:", perm_sel.selected_cols)
print("Shapes permutation sel:", X_tr_perm.shape, X_te_perm.shape, X_ba_perm.shape)

# %%
# 8) Inspección final
print("\nImportancias (train) por permutación:")
print(perm_sel.importances.sort_values(ascending=False).to_string())

print("\nPrimeras filas de X_train tras permutation_partitions:")
display(X_tr_perm.head())


/home/roma/proyecto_mlops/ambiente_python/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Shapes originales: X_train (235, 15) X_test (67, 15) X_back (34, 15)
✅ After rough filter: (235, 15) (67, 15) (34, 15)
→ cols after rough: ['EPS_A', 'EPS_B', 'EPS_C', 'EPS_D', 'EPS_A^2', 'EPS_A EPS_B', 'EPS_A EPS_C', 'EPS_A EPS_D', 'EPS_B^2', 'EPS_B EPS_C', 'EPS_B EPS_D', 'EPS_C^2', 'EPS_C EPS_D', 'EPS_D^2', 'Semana']
🔍 Frame sel cols: ['EPS_A', 'EPS_B']
Shapes frame sel: (235, 3) (67, 3) (34, 3)
✨ ABESS sel cols: ['EPS_A']
Shapes abess sel: (235, 2) (67, 2) (34, 2)
🌟 SHAP sel cols: ['EPS_A']
Shapes SHAP sel: (235, 2) (67, 2) (34, 2)
🔁 Permutation sel cols: ['EPS_A']
Shapes permutation sel: (235, 2) (67, 2) (34, 2)

Importancias (train) por permutación:
EPS_A    0.801199

Primeras filas de X_train tras permutation_partitions:


,EPS_A,Semana
0,-0.659360,2019-02-10
1,0.433301,2019-02-17
2,-0.659360,2019-02-24
3,0.253322,2019-03-03
4,0.968728,2019-03-10


In [4]:
# %% [markdown]
# # Test completo de pipeline M06__feature_selection: steps 1–7
# Asumimos que este notebook está en src/app.

# %%
import sys
from pathlib import Path

# 1) Asegura que 'src/app' esté en sys.path para poder importar nuestros módulos
APP_DIR = Path().resolve()           # .../mlops_canvas/src/app
if str(APP_DIR) not in sys.path:
    sys.path.append(str(APP_DIR))

# Para resolver la carpeta 'src' y luego 'data/processed/...'
SRC_DIR     = APP_DIR.parent        # .../mlops_canvas/src
EXPORT_DIR  = SRC_DIR / "data" / "processed" / "pipeline_selection"

# %%
# 2) Importa loader y todos los pasos
from m06__feature_selection.step01_import     import ParquetPartitionLoader2
from m06__feature_selection.step02_filtering  import filter_partitions
from m06__feature_selection.step03_frame      import frame_partitions
from m06__feature_selection.step04_abess      import abess_partitions
from m06__feature_selection.step05_shap_select import shap_partitions
from m06__feature_selection.step06_permutation import permutation_partitions
from m06__feature_selection.step07_export     import export_partitions

# %%
# 3) Carga tus particiones reales
loader = ParquetPartitionLoader2()
X_train, X_test, X_back, y_train, y_test, y_back = loader.load()
print("🚀 Original shapes:",
      "X_train", X_train.shape,
      "X_test",  X_test.shape,
      "X_back",  X_back.shape)

# %%
# 4) Step02: filtro rough
X_tr_f, X_te_f, X_ba_f, rough_tf = filter_partitions(
    X_train, X_test, X_back, y_train
)
print("✅ After rough filter shapes:", 
      X_tr_f.shape, X_te_f.shape, X_ba_f.shape)

# %%
# 5) Step03: selección forward + RFE (regresión)
from sklearn.linear_model import LinearRegression
X_tr_fr, X_te_fr, X_ba_fr, frame_sel = frame_partitions(
    X_tr_f, X_te_f, X_ba_f, y_train,
    estimator=LinearRegression(), forward_k=4, final_k=2
)
print("🔍 Frame selected cols:", frame_sel.selected_cols)
print("    Shapes:", X_tr_fr.shape, X_te_fr.shape, X_ba_fr.shape)

# %%
# 6) Step04: selección ABESS (regresión)
X_tr_ab, X_te_ab, X_ba_ab, abess_sel = abess_partitions(
    X_tr_fr, X_te_fr, X_ba_fr, y_train,
    mode="regression"
)
print("✨ ABESS selected cols:", abess_sel.selected_cols)
print("    Shapes:", X_tr_ab.shape, X_te_ab.shape, X_ba_ab.shape)

# %%
# 7) Step05: selección por SHAP (top 3, regresión)
X_tr_sh, X_te_sh, X_ba_sh, shap_sel = shap_partitions(
    X_tr_ab, X_te_ab, X_ba_ab, y_train,
    top_n=3, task="regression"
)
print("🌟 SHAP selected cols:", shap_sel.selected_cols)
print("    Shapes:", X_tr_sh.shape, X_te_sh.shape, X_ba_sh.shape)

# %%
# 8) Step06: selección por permutación (tol=0.01, R², 10 repeticiones)
X_tr_perm, X_te_perm, X_ba_perm, perm_sel = permutation_partitions(
    X_tr_sh, X_te_sh, X_ba_sh, y_train,
    tol=0.01, task="regression", scoring="r2", n_repeats=10
)
print("🔁 Permutation selected cols:", perm_sel.selected_cols)
print("    Shapes:", X_tr_perm.shape, X_te_perm.shape, X_ba_perm.shape)

# %%
# 9) Consulta importancias por permutación
print("\nPermutation importances (train):")
print(perm_sel.importances.sort_values(ascending=False).to_string())

# %%
# 10) Step07: exporta las particiones finales
export_partitions(X_tr_perm, X_te_perm, X_ba_perm)

# %%
# 11) Verifica que los archivos estén en 'src/data/processed/pipeline_selection'
print("\nArchivos en:", EXPORT_DIR)
print([p.name for p in EXPORT_DIR.iterdir()])

# %%
# 12) Inspección final: primeras filas de la partición entrenada final
print("\nPrimeras filas de X_train tras permutation_partitions:")
display(X_tr_perm.head())


🚀 Original shapes: X_train (235, 15) X_test (67, 15) X_back (34, 15)
✅ After rough filter shapes: (235, 15) (67, 15) (34, 15)
🔍 Frame selected cols: ['EPS_A', 'EPS_B']
    Shapes: (235, 3) (67, 3) (34, 3)
✨ ABESS selected cols: ['EPS_A']
    Shapes: (235, 2) (67, 2) (34, 2)
🌟 SHAP selected cols: ['EPS_A']
    Shapes: (235, 2) (67, 2) (34, 2)
🔁 Permutation selected cols: ['EPS_A']
    Shapes: (235, 2) (67, 2) (34, 2)

Permutation importances (train):
EPS_A    0.801199

Archivos en: /home/roma/proyecto_mlops/mlops_canvas/src/data/processed/pipeline_selection
['X_train_selected.parquet', 'X_test_selected.parquet', 'X_backtest_selected.parquet']

Primeras filas de X_train tras permutation_partitions:


,EPS_A,Semana
0,-0.659360,2019-02-10
1,0.433301,2019-02-17
2,-0.659360,2019-02-24
3,0.253322,2019-03-03
4,0.968728,2019-03-10


In [1]:
import sys
from pathlib import Path

# 1) Asegura que 'src/app' esté en sys.path
APP_DIR = Path().resolve()           # .../mlops_canvas/src/app
if str(APP_DIR) not in sys.path:
    sys.path.append(str(APP_DIR))

# Define ruta de exportación (debe coincidir con DataFrameExporter)
SRC_DIR    = APP_DIR.parent         # .../mlops_canvas/src
EXPORT_DIR = SRC_DIR / "data" / "processed" / "pipeline_selection"

print("🔧 Rutas configuradas:")
print(" APP_DIR    =", APP_DIR)
print(" EXPORT_DIR =", EXPORT_DIR)

# %%
# 2) Importa y ejecuta el pipeline completo
from m06__feature_selection import run_pipeline

X_train, X_test, X_back = run_pipeline()

# %%
# 3) Inspecciona shapes y columnas finales
print("🚀 Shapes finales:")
print(" X_train:", X_train.shape)
print(" X_test :", X_test.shape)
print(" X_back :", X_back.shape)

print("\n✨ Columnas seleccionadas finales:")
print(X_train.columns.tolist())

# %%
# 4) Vista previa de la partición de entrenamiento
display(X_train.head())

# %%
# 5) Revisa los archivos Parquet exportados
import os

print("📂 Archivos en directorio de exportación:")
if EXPORT_DIR.exists():
    for f in sorted(os.listdir(EXPORT_DIR)):
        print("  -", f)
else:
    print("  ⚠️ Directorio no existe:", EXPORT_DIR)

# %%
# 6) Carga uno de los Parquets para verificar su contenido
import pandas as pd

p_train = pd.read_parquet(EXPORT_DIR / "X_train_selected.parquet")
print("\n📑 Parquet 'X_train_selected.parquet' cargado, shape:", p_train.shape)
display(p_train.head())


🔧 Rutas configuradas:
 APP_DIR    = /home/roma/proyecto_mlops/mlops_canvas/src/app
 EXPORT_DIR = /home/roma/proyecto_mlops/mlops_canvas/src/data/processed/pipeline_selection


/home/roma/proyecto_mlops/ambiente_python/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pipeline finalizado. Variables seleccionadas finales:
['EPS_A', 'Semana']
🚀 Shapes finales:
 X_train: (235, 2)
 X_test : (67, 2)
 X_back : (34, 2)

✨ Columnas seleccionadas finales:
['EPS_A', 'Semana']


,EPS_A,Semana
0,-0.659360,2019-02-10
1,0.433301,2019-02-17
2,-0.659360,2019-02-24
3,0.253322,2019-03-03
4,0.968728,2019-03-10


📂 Archivos en directorio de exportación:
  - X_backtest_selected.parquet
  - X_test_selected.parquet
  - X_train_selected.parquet

📑 Parquet 'X_train_selected.parquet' cargado, shape: (235, 2)


,EPS_A,Semana
0,-0.659360,2019-02-10
1,0.433301,2019-02-17
2,-0.659360,2019-02-24
3,0.253322,2019-03-03
4,0.968728,2019-03-10


In [1]:
# %%
import sys
from pathlib import Path

# 1) Asegura que 'src/app' esté en sys.path
APP_DIR = Path().resolve()           # .../mlops_canvas/src/app
if str(APP_DIR) not in sys.path:
    sys.path.append(str(APP_DIR))

# 2) Importa tu función orquestadora
from m06__feature_selection.run import run_pipeline

# 3) Ejecuta todo el pipeline y muestra logs detallados
X_tr_all, X_te_all, X_ba_all = run_pipeline(technique="all", save_logs=True)

# 4) Comprueba shapes y primeras filas
print("→ Shapes finales:", X_tr_all.shape, X_te_all.shape, X_ba_all.shape)
display(X_tr_all.head())


/home/roma/proyecto_mlops/ambiente_python/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Original shapes: X_train (235, 15), X_test (67, 15), X_back (34, 15)
✔ [  filter   ] cols=15  t=0.013s
✔ [   frame   ] cols= 3  t=0.531s
✔ [   abess   ] cols= 2  t=0.024s
✔ [   shap    ] cols= 2  t=0.181s
✔ [permutation] cols= 2  t=0.265s

🔍 Reporte completo:
 • filter: 15 cols en 0.0131s
 • frame: 3 cols en 0.5307s
 • abess: 2 cols en 0.0237s
 • shap: 2 cols en 0.1805s
 • permutation: 2 cols en 0.265s

✨ Variables seleccionadas finales:
    ['EPS_A', 'Semana']
📦 Particiones exportadas.
→ Shapes finales: (235, 2) (67, 2) (34, 2)


,EPS_A,Semana
0,-0.659360,2019-02-17
1,0.433301,2019-02-24
2,-0.659360,2019-03-03
3,0.253322,2019-03-10
4,0.968728,2019-03-17


In [1]:
# %% llamada compuesta: filter → frame con logs
import pandas as pd          # ← Asegúrate de tener esto
from m06__feature_selection.run import run_pipeline

X_tr_ff, X_te_ff, X_ba_ff, logs = run_pipeline(
    techniques=["filter", "frame"],
    save_logs=True
)

# Ahora sí puedes verlos en un DataFrame
print(pd.DataFrame(logs))


/home/roma/proyecto_mlops/ambiente_python/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✔ Logs guardados en /home/roma/proyecto_mlops/mlops_canvas/src/output/log_selection/feature_selection_logs.json
Pipeline finalizado. Variables seleccionadas finales:
['EPS_A', 'EPS_B', 'Semana']
     step  time_sec  n_features
0  filter    0.0116          15
1   frame    0.4691           3
